# PR-0002 本番版 CPU Driveなし
下の1セルを実行すると、本番50反復・10,999,800記録を開始します。
初回のファイル選択では、点検済みPILOT_COMPLETE ZIPを1つ選んでください。
45分ごとのPARTIAL ZIPを保存し、同じランタイムでは同じセルを再実行してください。
新ランタイムではRESTORE_ZIPに最新PARTIAL ZIPのパス（またはUPLOAD）を設定します。
推定処理約2時間、余裕2〜3時間。DL・アップロード・操作待ちは別です。


In [ ]:
# Copyright 2026 Kentaro Sato -- Apache-2.0 (code); CC BY 4.0 (paper/card)
# PR-0002 MAIN RUN: paste this WHOLE file into ONE Colab code cell.
# Running this cell starts/resumes the fixed 50-repeat, 10,999,800-record run.
# The provided PILOT_COMPLETE ZIP passed saved-data acceptance on 2026-10-17.
MODE = "confirm"
RESTORE_ZIP = ""                # fresh runtime restore: "UPLOAD" or /content/latest_PARTIAL.zip
PILOT_ZIP = "UPLOAD"            # first run: choose the accepted PILOT_COMPLETE ZIP
CONFIRM_APPROVAL = "RUN PR-0002 CONFIRM"  # execute this cell only when ready to start
AUTO_DOWNLOAD = True
# No Drive, no pip install, CPU implementation; do not run two copies.
# Same runtime: keep RESTORE_ZIP blank; choose the same pilot ZIP again, or set
# PILOT_ZIP to its existing /content/...zip path to avoid another upload prompt.
# New runtime: set RESTORE_ZIP to the latest PARTIAL ZIP; PILOT_ZIP is then ignored.
# For >1 GB, upload via Colab Files and use its absolute path (less upload RAM).
# After restore, clear RESTORE_ZIP before the next SAME-runtime continuation.
# Each cell pauses safely after 45 min; save every PARTIAL ZIP locally.
# Four-hour cumulative budget and all scientific settings remain unchanged.
# Environment mismatches stop; do not bypass the gate or silently upgrade libraries.

PAYLOAD_SHA256 = '3f8eaf10ccbc0a961d32c6173ac39173ced377e9a128fdc90f9b1445e24a90bb'
CELL_ID = '568b5189bb5d58d5b1f4ba6f7497ebf234783d01e3bd3ebc90c543ca7ebcf80c'
PAYLOAD = '...'
import base64, hashlib, importlib, json, os, sys, zipfile, zlib
from pathlib import Path
if "COLAB_RELEASE_TAG" not in os.environ:
    try:
        import google.colab
    except ImportError as exc:
        raise RuntimeError("Run this cell in Colab. Local pilot/main execution is disabled.") from exc
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(key, "1")
raw = zlib.decompress(base64.b64decode(PAYLOAD))
if hashlib.sha256(raw).hexdigest() != PAYLOAD_SHA256:
    raise ValueError("Embedded source payload is corrupted")
bundle = json.loads(raw)
package = Path("/content") / ("pr0002_package_" + PAYLOAD_SHA256[:12])
package.mkdir(exist_ok=True)
for name, content in bundle.items():
    target = package / name
    if target.exists() and target.read_text() != content:
        raise ValueError("Existing package differs; use a clean runtime")
    target.write_text(content)
sys.path.insert(0, str(package))
for name in ("pr0002_checks", "fusion_confirmation_pr0002_v1", "covariance_consistency_colab_v1"):
    sys.modules.pop(name, None)
import fusion_confirmation_pr0002_v1 as runner
def select_zip(value):
    if value != "UPLOAD":
        return value or None
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select exactly one ZIP")
    filename = next(iter(uploaded))
    del uploaded
    return str(Path(filename).resolve())
restore_zip = select_zip(RESTORE_ZIP)
pilot_receipt = None
if MODE == "confirm":
    if CONFIRM_APPROVAL != "RUN PR-0002 CONFIRM":
        raise PermissionError("Main run is not authorized. Leave MODE='pilot' for now.")
    if restore_zip:
        with zipfile.ZipFile(restore_zip) as z:
            pilot_receipt = json.loads(z.read("manifest.json"))["pilot_receipt"]
    else:
        pilot_zip = select_zip(PILOT_ZIP)
        if pilot_zip:
            with zipfile.ZipFile(pilot_zip) as z:
                pilot_receipt = json.loads(z.read("pilot_receipt.json"))
        else:
            paths = list(Path("/content/pr0002_runs").glob("*_pilot_*/pilot_receipt.json"))
            matching = [json.loads(p.read_text()) for p in paths
                        if json.loads(p.read_text()).get("cell_payload_sha256") == CELL_ID]
            if len(matching) != 1:
                raise ValueError("Choose the matching completed pilot ZIP via PILOT_ZIP")
            pilot_receipt = matching[0]
RUN_RESULT = runner.run(MODE, "/content/pr0002_runs", bundle, CELL_ID,
    approval=CONFIRM_APPROVAL, pilot_receipt=pilot_receipt,
    restore_zip=restore_zip, auto_download=AUTO_DOWNLOAD)
if MODE == "pilot" and RUN_RESULT["status"] == "PILOT_COMPLETE":
    print("Pilot only completed. Share this ZIP for budget/acceptance review. No confirmation run was started.")
